# 🧠 PINN — Core Mechanic
## Gradient of Network OUTPUT w.r.t. its INPUT

---

### 📌 Why this matters
Normal training only needs:
```
∂Loss / ∂weights   → used by optimizer (Adam, SGD)
```
A PINN **additionally** needs:
```
∂u/∂x,  ∂u/∂t,  ∂²u/∂x²   → used inside the Physics Loss
```
Where `u` = network output, `x, t` = network inputs (space, time)

This notebook teaches you exactly **how to compute that in PyTorch.**

---
## 🔹 PART 1: Simple function first (not a network)
Let's verify with `u = x³`, where we **know** the answer:
- `du/dx = 3x²`
- `d²u/dx² = 6x`

In [ ]:
import torch

# requires_grad=True → tell PyTorch: track ALL operations on x
# (normally you only set this on weights — for PINNs, set it on INPUT too)
x = torch.tensor([2.0], requires_grad=True)
u = x ** 3

# ── First derivative: du/dx ──────────────────────────────────────────
du_dx = torch.autograd.grad(
    outputs=u,
    inputs=x,
    grad_outputs=torch.ones_like(u),  # needed because u is not a scalar
    create_graph=True                 # ⚠️ KEEP graph alive for 2nd derivative
)[0]

print(f"x        = {x.item()}")
print(f"u = x³   = {u.item()}")
print(f"du/dx    = {du_dx.item():<8}  ← expected 3x² = {3 * x.item()**2}  ✓")

# ── Second derivative: d²u/dx² ───────────────────────────────────────
# Just differentiate du_dx w.r.t. x again!
d2u_dx2 = torch.autograd.grad(
    outputs=du_dx,
    inputs=x,
    grad_outputs=torch.ones_like(du_dx),
    create_graph=True
)[0]

print(f"d²u/dx²  = {d2u_dx2.item():<8}  ← expected 6x  = {6 * x.item()}  ✓")

---
## 🔹 PART 2: Same trick on a real Neural Network

The **exact same** `torch.autograd.grad()` works when `u` is a network output.

⚠️ Use **Tanh** activation, NOT ReLU:
- ReLU's 2nd derivative = 0 almost everywhere → useless for physics loss
- Tanh is smooth and infinitely differentiable → perfect for PINNs

In [ ]:
# A tiny neural network: 1 input (x) → 1 output (u)
net = torch.nn.Sequential(
    torch.nn.Linear(1, 16),
    torch.nn.Tanh(),       # ✅ smooth activation
    torch.nn.Linear(16, 16),
    torch.nn.Tanh(),
    torch.nn.Linear(16, 1)
)

# Input: requires_grad=True on x (not just on weights!)
x2 = torch.tensor([[1.5]], requires_grad=True)
u2 = net(x2)

# Compute du/dx
du2_dx = torch.autograd.grad(
    u2, x2,
    grad_outputs=torch.ones_like(u2),
    create_graph=True
)[0]

# Compute d²u/dx²
d2u2_dx2 = torch.autograd.grad(
    du2_dx, x2,
    grad_outputs=torch.ones_like(du2_dx),
    create_graph=True
)[0]

print(f"Network output u(1.5)  = {u2.item():.4f}")
print(f"du/dx at x=1.5         = {du2_dx.item():.4f}")
print(f"d²u/dx² at x=1.5       = {d2u2_dx2.item():.4f}")
print()
print("ℹ️  Numbers are random (untrained network) — that's fine!")
print("   The point: we CAN compute ∂u/∂x and ∂²u/∂x² for ANY input.")
print("   This is exactly what goes into the PDE residual inside the physics loss.")

---
## 🔹 PART 3: Two inputs (x AND t) — what a real PINN uses

For the heat equation `∂u/∂t = α ∂²u/∂x²`, we need:
- `u` = network output (temperature)
- Input = `[x, t]` (space AND time)
- `∂u/∂t` from the input's time component
- `∂²u/∂x²` from the input's space component

In [ ]:
# Network: 2 inputs [x, t] → 1 output [u]
net2d = torch.nn.Sequential(
    torch.nn.Linear(2, 32),
    torch.nn.Tanh(),
    torch.nn.Linear(32, 32),
    torch.nn.Tanh(),
    torch.nn.Linear(32, 1)
)

# One sample point: x=0.5, t=0.1
xt = torch.tensor([[0.5, 0.1]], requires_grad=True)  # shape: (1, 2)
u3 = net2d(xt)

# Gradient w.r.t. BOTH inputs at once
grads = torch.autograd.grad(
    u3, xt,
    grad_outputs=torch.ones_like(u3),
    create_graph=True
)[0]  # shape: (1, 2)  → [∂u/∂x, ∂u/∂t]

du_dx = grads[:, 0:1]   # ∂u/∂x
du_dt = grads[:, 1:2]   # ∂u/∂t

# Second derivative w.r.t. x only
d2u_dx2 = torch.autograd.grad(
    du_dx, xt,
    grad_outputs=torch.ones_like(du_dx),
    create_graph=True
)[0][:, 0:1]   # take only the x-component

print(f"u(0.5, 0.1)  = {u3.item():.4f}")
print(f"∂u/∂x        = {du_dx.item():.4f}")
print(f"∂u/∂t        = {du_dt.item():.4f}")
print(f"∂²u/∂x²      = {d2u_dx2.item():.4f}")
print()
print("✅ Heat Equation Physics Residual would be:")
alpha = 0.01
residual = du_dt - alpha * d2u_dx2
print(f"   r = ∂u/∂t - α·∂²u/∂x²  = {residual.item():.4f}  (want this → 0 after training)")

---
## ✅ Summary — What you just learned

| Concept | Key Line |
|---|---|
| Track input gradients | `x = torch.tensor(..., requires_grad=True)` |
| 1st derivative | `torch.autograd.grad(u, x, ..., create_graph=True)` |
| 2nd derivative | Call `.grad()` again on the 1st derivative result |
| Why `create_graph=True` | Without it, graph is deleted after 1st call → 2nd derivative fails |
| Why Tanh not ReLU | ReLU's 2nd derivative = 0 almost everywhere → physics loss breaks |
| Physics residual | `r = ∂u/∂t - α·∂²u/∂x²` → push this to 0 during training |

---
### 🚀 Next: Day 2 — Full 1D Heat Equation PINN
We'll build:
- A proper network trained on the heat equation
- Boundary + initial condition losses
- Physics (PDE) loss
- Full training loop
- Plot comparing PINN prediction vs analytical solution